# 04 — Arazi Örtüsü Değişimi ve İklim İlişkisi

Bu notebook AS1–AS4 alt sorularının sayısal sonuçlarını üretir.

**Bölüm A — Haritalar (Earth Engine):**
1. Faz 3'te seçilen modelin ayarlarıyla Earth Engine'de bir sınıflandırıcı eğitilir (karar A). Bu model Colab'daki en iyi modelle **aynı test noktalarında** karşılaştırılır.
2. 2014 ve 2024 kompozitleri 30 m'de sınıflandırılır. Sonuç 1 km hücrelerde sınıf oranları olarak Drive'a yazılır. İl il çalışılır, önce Artvin pilot olarak çalışır ve kota gösterilir.
3. Geçiş matrisi (2014'te X olan piksel 2024'te ne oldu?) il başına 20.000 rastgele pikselden tahmin edilir.

**Bölüm B — Analiz (Colab, Earth Engine yok):**
4. Sınıf alanları ve değişim (%).
5. MODIS LST, ERA5-Land ve CHIRPS şehir serilerinde Mann-Kendall trend testi ve Sen eğimi.
6. 1 km hücrelerde Δ(arazi örtüsü) ile ΔLST ve ΔNDMI arasında Spearman korelasyonu.
7. LST regresyonu (Random Forest Regressor, mekânsal blok CV, SHAP).
8. Kentsel ısı adası (İstanbul, İzmir).
9. Alt soru özet tablosu.

> **Önemli yorum kuralı:** Bütün bulgular **ilişki (korelasyon)** düzeyindedir. "Yapılaşma sıcaklığı artırdı" değil, "yapılaşmanın arttığı hücrelerde sıcaklık artışı daha fazla" deriz.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) özel GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Özel repoyu okumak için Colab'ın **Gizli anahtarlar (Secrets)** bölümündeki `GITHUB_TOKEN` kullanılır. Token ekrana yazdırılmaz.

In [ ]:
!pip install -q earthengine-api geemap pymannkendall shap

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git hatası: " + (r.stderr or r.stdout).replace(token, "***"))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        raise RuntimeError("Colab'da sol menüdeki anahtar simgesinden GITHUB_TOKEN ekleyin ve "
                           "'Not defteri erişimi'ni açın (README'ye bakın).") from e
    if os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["-C", CODE_DIR, "pull", "-q"], token)
    else:
        _git(["clone", "-q", f"https://{token}@github.com/{GITHUB_REPO}.git", CODE_DIR], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import ee
from src import gee_utils as gu
importlib.reload(gu)
gu.init_ee()

In [ ]:
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import ml_utils as mu
importlib.reload(mu)

FORCE_RECOMPUTE = False
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
T, F, M = config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR
RAW_GEE = f"{config.DATA_RAW}/gee"
os.makedirs(RAW_GEE, exist_ok=True)
C = config.COLLECTIONS
CITY_LIST = list(config.CITIES)
LABELS = sorted(config.CLASSES)
CLASS_NAMES = [config.CLASSES[k] for k in LABELS]
Y0, Y1 = config.CHANGE_YEARS
FEATS = config.MODEL_FEATURES

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

def set_props(props):
    return lambda f: f.set(props)

geoms = {c: gu.city_geometry(c) for c in CITY_LIST}
grid_proj = ee.Projection(config.GRID_CRS).atScale(config.GRID_SIZE_M)

samples = pd.read_csv(f"{config.DATA_PROCESSED}/samples_2021.csv")
split = pd.read_csv(f"{config.DATA_PROCESSED}/split_assignments.csv")[["sample_id", "split"]]
samples = samples.merge(split, on="sample_id")
meta = json.load(open(f"{M}/best_model.json", encoding="utf-8"))
print("Faz 3 en iyi model:", meta["best_model"])

## A1. Earth Engine sınıflandırıcısı
Earth Engine'in hazır sınıflandırıcıları scikit-learn ile birebir aynı değildir. Eşleme şöyle yapılıyor:
- En iyi model **Random Forest** ise `smileRandomForest` (aynı ağaç sayısı, yaprak büyüklüğü ve her bölünmede denenen özellik sayısı).
- En iyi model **XGBoost** ise `smileGradientTreeBoost` (aynı ağaç sayısı, öğrenme oranı, alt örnekleme; derinlik → en fazla yaprak sayısı).
- En iyi model **SVM veya MLP** ise Earth Engine'de birebir karşılığı olmadığı için doğrulama kümesinde en iyi olan **ağaç modeli** kullanılır ve bu açıkça yazılır.

Sınıflandırıcı, Faz 3'teki **eğitim** noktalarıyla eğitiliyor. Noktaların özellik değerleri zaten tabloda olduğu için Earth Engine'de görüntüden yeniden örnekleme yapılmıyor, bu yüzden kota harcanmıyor.

In [ ]:
val = pd.read_csv(f"{T}/model_karsilastirma_dogrulama.csv", index_col=0)
EE_MODEL = meta["best_model"] if meta["best_model"] in ("RandomForest", "XGBoost") \
    else val.loc[["RandomForest", "XGBoost"], "f1_macro"].idxmax()
ee_params = meta["all_params"][EE_MODEL]
print("Earth Engine'de kullanılan model:", EE_MODEL, ee_params,
      "" if EE_MODEL == meta["best_model"] else f"(Colab'daki en iyi model {meta['best_model']} GEE'de yok)")

train = samples[samples["split"] == "train"]
test = samples[samples["split"] == "test"]
train_fc = gu.df_to_fc(train, ["class"] + FEATS)
ee_clf = gu.make_ee_classifier(EE_MODEL, ee_params).train(
    features=train_fc, classProperty="class", inputProperties=FEATS)

## A2. Earth Engine modeli ile Colab modeli: aynı test noktalarında karşılaştırma
Haritaları Earth Engine modeli üreteceği için bu modelin Faz 3'teki modelle benzer başarıda olduğunu göstermemiz gerekir. Test noktaları Earth Engine'de sınıflandırılıyor; Accuracy, macro-F1 ve Kappa yan yana konuyor. Hesap tablodaki değerler üzerinden yapıldığı için hafif.

macro-F1 farkı %2 puandan büyükse notebook uyarı verir.

In [ ]:
cmp_csv = f"{T}/gee_vs_colab_karsilastirma.csv"
pred_csv = f"{config.DATA_PROCESSED}/test_predictions_gee.csv"
if os.path.exists(pred_csv) and not FORCE_RECOMPUTE:
    gee_pred = pd.read_csv(pred_csv)
else:
    test_fc = gu.df_to_fc(test, ["sample_id"] + FEATS)
    res = test_fc.classify(ee_clf, "pred")
    got = ee.Dictionary({"id": res.aggregate_array("sample_id"), "pred": res.aggregate_array("pred")}).getInfo()
    ids, out = got["id"], got["pred"]
    gee_pred = pd.DataFrame({"sample_id": ids, "pred_gee": out})
    gee_pred.to_csv(pred_csv, index=False)

colab = pd.read_csv(f"{config.DATA_PROCESSED}/test_predictions.csv")
both = colab.merge(gee_pred, on="sample_id")
rows = []
for label, col in [(f"Colab {meta['best_model']}", f"pred_{meta['best_model']}"),
                   (f"Colab {EE_MODEL}", f"pred_{EE_MODEL}"),
                   (f"Earth Engine {EE_MODEL}", "pred_gee")]:
    m = mu.compute_metrics(both["class"], both[col], None, LABELS)
    rows.append({"model": label, "accuracy": m["accuracy"], "f1_macro": m["f1_macro"], "kappa": m["kappa"]})
gee_cmp = pd.DataFrame(rows).drop_duplicates("model").set_index("model").round(4)
gee_cmp["n_test"] = len(both)
gee_cmp["harita_uyumu_colab_en_iyi"] = (both["pred_gee"] == both[f"pred_{meta['best_model']}"]).mean().round(4)
gee_cmp.to_csv(cmp_csv, encoding="utf-8-sig")
display(gee_cmp)

diff = gee_cmp.loc[f"Colab {meta['best_model']}", "f1_macro"] - gee_cmp.loc[f"Earth Engine {EE_MODEL}", "f1_macro"]
GEE_OK = abs(diff) <= config.GEE_VS_COLAB_WARN
print(f"macro-F1 farkı (Colab en iyi − GEE): {diff:+.4f} →",
      "kabul edilebilir" if GEE_OK else "BÜYÜK FARK: bu çıktıyı bize gönderin, haritalara geçmeden konuşalım.")

## A3. 2014 ve 2024 sınıflandırması → 1 km sınıf oranları (il il)
Her il ve yıl için Earth Engine 30 m kompoziti sınıflandırıyor, sonra her 1 km hücrede her sınıfın oranını hesaplıyor (`reduceResolution`). Hücreler Faz 1 grid'iyle aynı (`x, y`), böylece iklim değişkenleriyle doğrudan birleşiyor. **İl alanları da bu oranlardan hesaplanıyor**, yani ayrı bir alan görevi yok.

Bellek için Faz 1'deki ayarlar kullanılıyor: görev başına tek yıl, 32 bit, `tileScale=16`.

**Geçiş matrisi:** 2014 ve 2024 sınıfını aynı pikselde görmek için iki yılın da sınıflandırılmış olması gerekir. Tüm ili iki yılla birlikte işlemek ağır olduğundan, il başına **20.000 rastgele piksel** örnekleniyor. Geçiş oranları bu örnekten tahmin ediliyor. Bu, alan tahmininde yaygın bir istatistiksel yöntemdir; 20.000 noktada bir oranın hata payı en fazla yaklaşık ±0.7 puandır.

**Model belirsizliği:** Model 2021 etiketiyle eğitildi. 2014 ve 2024'e uygulandığında doğruluğunu ölçemiyoruz, çünkü o yıllar için etiket yok. Yıllar arası atmosfer ve fenoloji farkları, gerçek değişim olmasa da sınıf değişimi gibi görünebilir. Bu bir sınırlılık maddesidir. Küçük değişimleri (birkaç puan) temkinli yorumlayın.

In [ ]:
GRID_TILE_SCALE = 16

def classified(city, y):
    return gu.classification_composite(geoms[city], y).classify(ee_clf).rename("lc")

def lc_grid_fc(city, y):
    lc = classified(city, y)
    onehot = ee.Image.cat([lc.eq(k).rename(f"lc{k}_{y}") for k in LABELS]
                          + [lc.mask().unmask(0).rename(f"lc_valid_{y}")]).toFloat()
    img = (onehot.setDefaultProjection(crs=config.GRID_CRS, scale=config.LANDSAT_SCALE_M)
           .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=2048).reproject(grid_proj)
           .addBands(ee.Image.pixelCoordinates(grid_proj)))
    return img.sample(region=geoms[city], projection=grid_proj, geometries=False,
                      dropNulls=False, tileScale=GRID_TILE_SCALE).map(set_props({"city": city}))

def lc_cols(y):
    return ["city", "x", "y"] + [f"lc{k}_{y}" for k in LABELS] + [f"lc_valid_{y}"]

def transitions_fc(city):
    img = classified(city, Y0).rename(f"c{Y0}").addBands(classified(city, Y1).rename(f"c{Y1}"))
    return img.sample(region=geoms[city], scale=config.LANDSAT_SCALE_M, numPixels=config.TRANSITION_POINTS,
                      seed=config.RANDOM_STATE, tileScale=GRID_TILE_SCALE, geometries=False
                      ).map(set_props({"city": city}))

def city_exports(city):
    ex = {f"lc_grid_{city}_{y}": (lc_grid_fc(city, y), lc_cols(y)) for y in config.CHANGE_YEARS}
    ex[f"transitions_{city}"] = (transitions_fc(city), ["city", f"c{Y0}", f"c{Y1}"])
    return ex

### Pilot: Artvin
Kota kullanımı gösteriliyor. Kalan 6 il için kaba tahmin aylık kotanın %30'unu aşmazsa diğer iller otomatik başlar. A2'deki karşılaştırmada büyük fark çıkarsa hiç başlamaz.

In [ ]:
pilot = config.PILOT_CITY
KOTA_CSV = f"{T}/gee_kota_kullanimi.csv"
if not GEE_OK:
    raise RuntimeError("GEE ve Colab modelleri arasında büyük fark var (A2). Haritalara geçilmedi.")

pilot_usage = gu.run_exports(city_exports(pilot), RAW_GEE, force=FORCE_RECOMPUTE)
if pilot_usage is not None:
    display(pilot_usage)

kota = pd.read_csv(KOTA_CSV)
names = set(city_exports(pilot))
pk = kota[kota["task"].isin(names) & (kota["state"] == "COMPLETED")].drop_duplicates("task", keep="last")
pilot_eecu = pd.to_numeric(pk["eecu_saat"], errors="coerce").sum()
km2 = {c: geoms[c].area(1000).divide(1e6).getInfo() for c in CITY_LIST}
est = pilot_eecu * sum(km2[c] for c in CITY_LIST if c != pilot) / km2[pilot]
used = pd.to_numeric(kota["eecu_saat"], errors="coerce").sum()
print(f"{pilot} (Faz 4): {pilot_eecu:.3f} EECU-saat | kalan 6 il KABA tahmin: {est:.2f} EECU-saat "
      f"(%{100 * est / config.GEE_MONTHLY_EECU_HOURS:.1f}) | şimdiye kadar toplam kayıtlı: {used:.2f}")
if names - set(pk["task"]) or pilot_eecu == 0:
    PILOT_OK = False
    print("UYARI: Pilot kota kullanımı kayıtta eksik. Tasks sayfasına bakın; makulse sonraki hücrede PILOT_OK = True yapın.")
else:
    PILOT_OK = (est <= config.PILOT_MAX_QUOTA_FRACTION * config.GEE_MONTHLY_EECU_HOURS
                and used + est <= 0.9 * config.GEE_MONTHLY_EECU_HOURS)
    print("→ Kalan iller otomatik başlatılacak." if PILOT_OK else
          "DURDU: Tahmin kota sınırını aşıyor. Bu çıktıyı bize gönderin.")

In [ ]:
STOP_AFTER_PILOT = False
if STOP_AFTER_PILOT or not PILOT_OK:
    print("Kalan iller başlatılmadı.")
else:
    rest = {}
    for city in CITY_LIST:
        if city != pilot:
            rest.update(city_exports(city))
    usage = gu.run_exports(rest, RAW_GEE, force=FORCE_RECOMPUTE)
    if usage is not None:
        display(usage)

### Değişim haritaları (görsel)
Rapor için her ilin 2014 ve 2024 sınıf haritası ile **değişim haritası** küçük boyutlu görüntü olarak indiriliyor. Değişim haritasında yalnızca sınıfı değişen pikseller boyanıyor (2024'teki yeni sınıfının rengiyle), değişmeyenler gri gösteriliyor. Görüntüler 1024 piksel genişliğinde olduğundan Earth Engine bunları kaba çözünürlükte hesaplar. Bunlar görsel amaçlıdır; sayısal sonuçlar yukarıdaki 30 m görevlerinden gelir.

In [ ]:
import requests, matplotlib.image as mpimg
from matplotlib.patches import Patch
map_dir = f"{F}/lc_maps"
os.makedirs(map_dir, exist_ok=True)
palette = [config.CLASS_COLORS[k].lstrip("#") for k in LABELS]

def thumb(img, city, path, vis):
    if os.path.exists(path) and not FORCE_RECOMPUTE:
        return
    url = img.clip(geoms[city]).visualize(**vis).getThumbURL(
        {"region": geoms[city].bounds(), "dimensions": 1024, "format": "png"})
    with open(path, "wb") as f:
        f.write(requests.get(url, timeout=600).content)

for city in CITY_LIST:
    c0, c1 = classified(city, Y0), classified(city, Y1)
    vis = {"min": 0, "max": len(LABELS) - 1, "palette": palette}
    thumb(c0, city, f"{map_dir}/{city}_{Y0}.png", vis)
    thumb(c1, city, f"{map_dir}/{city}_{Y1}.png", vis)
    changed = c1.updateMask(c0.neq(c1))
    base = ee.Image(0).visualize(palette=["e0e0e0"])
    chg = ee.ImageCollection([base, changed.visualize(**vis)]).mosaic()
    thumb(chg, city, f"{map_dir}/{city}_degisim.png", {})

fig, axes = plt.subplots(3, len(CITY_LIST), figsize=(3.2 * len(CITY_LIST), 10))
for j, city in enumerate(CITY_LIST):
    for i, (suffix, title) in enumerate([(Y0, str(Y0)), (Y1, str(Y1)), ("degisim", "değişim")]):
        axes[i, j].imshow(mpimg.imread(f"{map_dir}/{city}_{suffix}.png"))
        axes[i, j].set_title(f"{city} {title}", fontsize=9); axes[i, j].axis("off")
fig.legend(handles=[Patch(color=config.CLASS_COLORS[k], label=config.CLASSES[k]) for k in LABELS]
           + [Patch(color="#e0e0e0", label="değişmedi")], loc="lower center", ncol=7, fontsize=9)
plt.subplots_adjust(bottom=0.06)
savefig("degisim_haritalari")

# Bölum B — Analiz
## B1. Sınıf alanları ve değişim (2014 → 2024)
Alan = Σ (hücredeki sınıf oranı × 1 km²). Değişim iki şekilde veriliyor:
- **Puan farkı (pp):** İlin yüzde kaçının o sınıf olduğu, 2014'ten 2024'e kaç puan değişti.
- **Göreli değişim (%):** Sınıfın kendi alanına göre değişim. Küçük sınıflarda (ör. su) göreli değişim büyük görünebilir, iki ölçüye birlikte bakın.

In [ ]:
def load_lc(city):
    d = None
    for y in config.CHANGE_YEARS:
        part = pd.read_csv(f"{RAW_GEE}/lc_grid_{city}_{y}.csv")
        d = part if d is None else d.merge(part, on=["city", "x", "y"], how="outer")
    return d

lc = pd.concat([load_lc(c) for c in CITY_LIST], ignore_index=True)
grid = pd.read_csv(f"{config.DATA_PROCESSED}/grid_1km_2014_2024.csv")
cells = grid.merge(lc, on=["city", "x", "y"], how="left")
cells.to_csv(f"{config.DATA_PROCESSED}/grid_1km_lc_iklim.csv", index=False)
CELL_KM2 = (config.GRID_SIZE_M / 1000) ** 2

rows = []
for city, d in cells.groupby("city"):
    for k in LABELS:
        a0, a1 = d[f"lc{k}_{Y0}"].sum() * CELL_KM2, d[f"lc{k}_{Y1}"].sum() * CELL_KM2
        tot0 = d[[f"lc{j}_{Y0}" for j in LABELS]].sum().sum() * CELL_KM2
        tot1 = d[[f"lc{j}_{Y1}" for j in LABELS]].sum().sum() * CELL_KM2
        rows.append({"city": city, "sinif": config.CLASSES[k], f"alan_{Y0}_km2": a0, f"alan_{Y1}_km2": a1,
                     f"pay_{Y0}_yuzde": 100 * a0 / tot0, f"pay_{Y1}_yuzde": 100 * a1 / tot1,
                     "degisim_km2": a1 - a0, "degisim_pp": 100 * (a1 / tot1 - a0 / tot0),
                     "goreli_degisim_yuzde": 100 * (a1 - a0) / a0 if a0 > 0 else np.nan})
areas = pd.DataFrame(rows).round(3)
areas.to_csv(f"{T}/sinif_alanlari_degisim.csv", index=False, encoding="utf-8-sig")

pp = areas.pivot(index="city", columns="sinif", values="degisim_pp").reindex(index=CITY_LIST, columns=CLASS_NAMES)
plt.figure(figsize=(10, 5))
sns.heatmap(pp, annot=True, fmt="+.2f", cmap="RdBu_r", center=0, cbar_kws={"label": "puan (pp)"})
plt.title(f"Sınıf payındaki değişim, {Y0} → {Y1} (yüzde puan)"); plt.ylabel("")
savefig("sinif_degisim_pp")

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, y in zip(axes, config.CHANGE_YEARS):
    areas.pivot(index="city", columns="sinif", values=f"pay_{y}_yuzde").reindex(index=CITY_LIST, columns=CLASS_NAMES) \
        .plot(kind="bar", stacked=True, ax=ax, color=[config.CLASS_COLORS[k] for k in LABELS], width=0.8, legend=False)
    ax.set_title(f"{y}"); ax.set_ylabel("İl alanının yüzdesi (%)"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
savefig("sinif_alanlari_2014_2024")
areas[areas["city"].isin(CITY_LIST)].head(12)

## B2. Geçiş matrisleri
Satır = 2014 sınıfı, sütun = 2024 sınıfı. Hücredeki değer, 2014'te o sınıfta olan piksellerin yüzde kaçının 2024'te sütundaki sınıfta olduğunu gösterir. Köşegen "değişmedi" demektir. Tahmini alan (km²) için örnek oranı il alanıyla çarpılıyor.

In [ ]:
trans = pd.concat([pd.read_csv(f"{RAW_GEE}/transitions_{c}.csv") for c in CITY_LIST], ignore_index=True)
city_area = cells.groupby("city").size() * CELL_KM2
tm_rows = []
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, city in zip(axes.ravel(), CITY_LIST):
    d = trans[trans["city"] == city]
    ct = pd.crosstab(d[f"c{Y0}"], d[f"c{Y1}"]).reindex(index=LABELS, columns=LABELS, fill_value=0)
    row_pct = 100 * ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0)
    km2_est = ct / ct.values.sum() * city_area[city]
    for i in LABELS:
        for j in LABELS:
            tm_rows.append({"city": city, "sinif_2014": config.CLASSES[i], "sinif_2024": config.CLASSES[j],
                            "n_nokta": int(ct.loc[i, j]), "satir_yuzde": row_pct.loc[i, j],
                            "tahmini_km2": km2_est.loc[i, j]})
    sns.heatmap(row_pct, annot=True, fmt=".0f", cmap="Greens", vmin=0, vmax=100, cbar=False, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={"size": 7})
    ax.set_title(f"{city} (n={len(d)})"); ax.set_xlabel(str(Y1)); ax.set_ylabel(str(Y0))
    ax.tick_params(axis="x", rotation=40, labelsize=7); ax.tick_params(axis="y", labelsize=7)
axes.ravel()[-1].axis("off")
savefig("gecis_matrisleri")
tm = pd.DataFrame(tm_rows).round(3)
tm.to_csv(f"{T}/gecis_matrisi.csv", index=False, encoding="utf-8-sig")

## B3. Şehir ölçeğinde iklim trendleri (2003–2024)
**Mann-Kendall testi:** Bir zaman serisinde artan ya da azalan **monoton bir eğilim** olup olmadığını sınar. Veri normal dağılmak zorunda değildir ve aykırı yıllara karşı dayanıklıdır; iklim serilerinde en yaygın trend testidir. p < 0.05 ise eğilim istatistiksel olarak anlamlı kabul edilir.

**Sen eğimi:** Trendin büyüklüğüdür (birim/yıl). Tüm yıl çiftleri arasındaki eğimlerin medyanıdır, bu yüzden tek bir sıcak yıldan etkilenmez.

ERA5-Land toprak nemi ~11 km çözünürlükte olduğu için **yalnızca burada**, şehir ölçeğinde kullanılıyor.

In [ ]:
import pymannkendall as mk
climate = pd.read_csv(f"{config.DATA_PROCESSED}/iklim_sehir_yillik.csv")
VARS = {"modis_lst_day": "MODIS yaz gündüz LST (°C)", "modis_lst_night": "MODIS yaz gece LST (°C)",
        "era5_t2m_summer": "ERA5 yaz 2 m sıcaklık (°C)", "era5_sm_summer": "ERA5 yaz toprak nemi (m³/m³)",
        "era5_sm_annual": "ERA5 yıllık toprak nemi (m³/m³)", "chirps_precip_annual": "CHIRPS yıllık yağış (mm)"}
rows = []
for city, d in climate.groupby("city"):
    d = d.sort_values("year")
    for v, label in VARS.items():
        s = d[v].dropna()
        r = mk.original_test(s.values)
        rows.append({"city": city, "degisken": v, "n_yil": len(s), "trend": r.trend, "p": r.p,
                     "sen_egimi_yil": r.slope, "sen_egimi_10yil": 10 * r.slope, "tau": r.Tau})
trend = pd.DataFrame(rows).round(5)
trend.to_csv(f"{T}/iklim_trend_mann_kendall.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, (v, label) in zip(axes.ravel(), VARS.items()):
    for city in CITY_LIST:
        d = climate[climate["city"] == city].sort_values("year")
        ax.plot(d["year"], d[v], marker="o", ms=3, lw=1.2, label=city)
    ax.set_title(label, fontsize=10); ax.set_xlabel("")
axes[0, 0].legend(fontsize=7, ncol=2)
savefig("iklim_serileri")

sig = trend.pivot(index="city", columns="degisken", values="sen_egimi_10yil").reindex(index=CITY_LIST, columns=list(VARS))
pv = trend.pivot(index="city", columns="degisken", values="p").reindex(index=CITY_LIST, columns=list(VARS))
stars = pd.DataFrame(np.where(pv.values < 0.05, "*", ""), index=sig.index, columns=sig.columns)
annot = sig.round(3).astype(str) + stars
plt.figure(figsize=(12, 4.5))
sns.heatmap(sig.div(sig.abs().max()), annot=annot, fmt="", cmap="RdBu_r", center=0, cbar=False)
plt.title("Sen eğimi (10 yılda değişim); * = Mann-Kendall p < 0.05  (renk: sütun içinde ölçekli)")
plt.ylabel(""); plt.xlabel("")
savefig("iklim_trend_ozet")
trend[trend["degisken"] == "modis_lst_day"]

## B4. 1 km hücrelerde değişim–iklim ilişkisi (Spearman)
Her hücre için 2024 − 2014 farkı (Δ) hesaplanıyor:
- **Arazi örtüsü:** Δorman, Δtarım, Δyapılaşma oranları (model haritalarından), Δsu (MNDWI su oranı; Faz 1'deki karar gereği iki yılda da aynı yöntem).
- **İndeks:** ΔNDVI.
- **İklim göstergesi:** ΔLST (Landsat, 30 m'den 1 km'ye ortalama), ΔMODIS LST, ΔNDMI (nem göstergesi).

**Spearman korelasyonu:** Pearson'dan farklı olarak değerlerin **sıralarını** kullanır. Doğrusal olmayan ama tek yönlü ilişkileri de yakalar ve aykırı değerlere dayanıklıdır.

**Yıl etkisi:** 2014 yazı ile 2024 yazı farklı hava koşullarına sahip olabilir. Bir ildeki tüm hücreleri aynı miktarda ısıtan bir fark Spearman korelasyonunu değiştirmez, çünkü korelasyon hücreleri **birbirine göre** karşılaştırır.

**p-değerleri hakkında:** Komşu hücreler birbirine benzer (mekânsal otokorelasyon). Hücreler bağımsız olmadığı için p-değerleri olduğundan küçük çıkar. Bu yüzden p'den çok **r'nin büyüklüğüne ve yönüne** bakın. Bu durum sınırlılıklarda yazılı.

Yalnızca iki yılda da sıcaklık verisinin en az %50'si olan ve sınıflandırılmış hücreler kullanılıyor.

In [ ]:
from scipy.stats import spearmanr
c = cells.copy()
ok = (c[f"st_valid_{Y0}"] >= 0.5) & (c[f"st_valid_{Y1}"] >= 0.5) & (c[f"lc_valid_{Y0}"] > 0.5) & (c[f"lc_valid_{Y1}"] > 0.5)
c = c[ok].copy()
c["d_orman"] = c[f"lc0_{Y1}"] - c[f"lc0_{Y0}"]
c["d_tarim"] = c[f"lc1_{Y1}"] - c[f"lc1_{Y0}"]
c["d_yapi"] = c[f"lc2_{Y1}"] - c[f"lc2_{Y0}"]
c["d_su"] = c[f"mndwi_water_{Y1}"] - c[f"mndwi_water_{Y0}"]
c["d_ndvi"] = c[f"NDVI_{Y1}"] - c[f"NDVI_{Y0}"]
c["d_lst"] = c[f"LST_{Y1}"] - c[f"LST_{Y0}"]
c["d_modis_lst"] = c[f"modis_lst_day_{Y1}"] - c[f"modis_lst_day_{Y0}"]
c["d_ndmi"] = c[f"NDMI_{Y1}"] - c[f"NDMI_{Y0}"]
c.to_csv(f"{config.DATA_PROCESSED}/grid_1km_delta.csv", index=False)

DRIVERS = {"d_orman": "Δorman", "d_tarim": "Δtarım", "d_yapi": "Δyapılaşma", "d_su": "Δsu", "d_ndvi": "ΔNDVI"}
TARGETS = {"d_lst": "ΔLST (Landsat)", "d_modis_lst": "ΔLST (MODIS)", "d_ndmi": "ΔNDMI (nem)"}
rows = []
for city, d in c.groupby("city"):
    for x, xl in DRIVERS.items():
        for y, yl in TARGETS.items():
            dd = d[[x, y]].dropna()
            if len(dd) < 30 or dd[x].std() == 0:
                continue
            r, p = spearmanr(dd[x], dd[y])
            rows.append({"city": city, "degisken": xl, "hedef": yl, "n_hucre": len(dd), "spearman_r": r, "p": p})
sp = pd.DataFrame(rows).round(4)
sp.to_csv(f"{T}/spearman_degisim_iklim.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
for ax, (y, yl) in zip(axes, TARGETS.items()):
    piv = sp[sp["hedef"] == yl].pivot(index="city", columns="degisken", values="spearman_r") \
            .reindex(index=CITY_LIST, columns=list(DRIVERS.values()))
    sns.heatmap(piv, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-0.6, vmax=0.6, ax=ax, cbar=ax is axes[-1])
    ax.set_title(yl); ax.set_xlabel(""); ax.set_ylabel("")
savefig("spearman_isi_haritasi")
print("Hücre sayıları:", c.groupby("city").size().to_dict())

In [ ]:
# Örnek dağılım grafikleri: alt sorulara en doğrudan bağlı ilişkiler
pairs = [("Artvin", "d_orman", "d_lst"), ("Sanliurfa", "d_ndvi", "d_lst"), ("Istanbul", "d_yapi", "d_lst"),
         ("Izmir", "d_yapi", "d_lst"), ("Konya", "d_su", "d_lst"), ("Konya", "d_tarim", "d_ndmi")]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, (city, x, y) in zip(axes.ravel(), pairs):
    d = c[c["city"] == city][[x, y]].dropna()
    ax.hexbin(d[x], d[y], gridsize=40, cmap="viridis", mincnt=1, bins="log")
    r, p = spearmanr(d[x], d[y]) if len(d) > 2 else (np.nan, np.nan)
    ax.set_title(f"{city}: ρ = {r:.2f} (n={len(d)})", fontsize=10)
    ax.set_xlabel({**DRIVERS, **TARGETS}[x]); ax.set_ylabel({**DRIVERS, **TARGETS}[y])
    ax.axhline(0, color="w", lw=0.6); ax.axvline(0, color="w", lw=0.6)
savefig("degisim_iklim_dagilim")

## B5. LST regresyonu: hangi değişken yüzey sıcaklığıyla en çok ilişkili?
**Hedef:** 2024 yazı Landsat LST (her 1 km hücre). **Girdiler:** 2024 arazi örtüsü oranları (6 sınıf), NDVI, NDBI, MNDWI, NDMI ve rakım. Model: **Random Forest Regressor**. Karşılaştırma için doğrusal regresyon da kuruluyor.

**Mekânsal blok çapraz doğrulama:** Komşu hücreler birbirine çok benzer. Rastgele bölmede test hücrelerinin komşuları eğitimde kalır ve başarı yapay olarak yüksek çıkar (mekânsal veri sızıntısı). Bu yüzden harita 10 km × 10 km bloklara bölünüyor ve **aynı bloktaki hücreler hep aynı katmana** düşüyor (GroupKFold, 5 katman).

**Metrikler:**
- **R²:** LST'deki değişkenliğin yüzde kaçı açıklanıyor (1 = tamamı).
- **RMSE:** Tipik hata (°C). Büyük hataları daha çok cezalandırır.
- **MAE:** Ortalama mutlak hata (°C).

**SHAP:** Hangi girdinin tahmini ne yönde ve ne kadar etkilediğini gösterir.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, cross_validate

REG_X = [f"lc{k}_{Y1}" for k in LABELS] + [f"{b}_{Y1}" for b in ["NDVI", "NDBI", "MNDWI", "NDMI"]] + ["elevation"]
REG_NAMES = {**{f"lc{k}_{Y1}": f"{config.CLASSES[k]} oranı" for k in LABELS},
             **{f"{b}_{Y1}": b for b in ["NDVI", "NDBI", "MNDWI", "NDMI"]}, "elevation": "Rakım"}
r = cells[(cells[f"st_valid_{Y1}"] >= 0.5) & (cells[f"lc_valid_{Y1}"] > 0.5)].dropna(subset=REG_X + [f"LST_{Y1}"])
Xr, yr = r[REG_X], r[f"LST_{Y1}"]
blocks = (r["city"] + "_" + (r["x"] // config.SPATIAL_BLOCK_M).astype(int).astype(str) + "_"
          + (r["y"] // config.SPATIAL_BLOCK_M).astype(int).astype(str))
gkf = GroupKFold(n_splits=5)
scoring = {"r2": "r2", "rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error"}
regs = {"RandomForestRegressor": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1,
                                                       random_state=config.RANDOM_STATE),
        "LinearRegression": make_pipeline(StandardScaler(), LinearRegression())}
rows = []
for name, reg in regs.items():
    cvr = cross_validate(reg, Xr, yr, groups=blocks, cv=gkf, scoring=scoring, n_jobs=1)
    rows.append({"model": name, "R2_ort": cvr["test_r2"].mean(), "R2_std": cvr["test_r2"].std(),
                 "RMSE_ort_C": -cvr["test_rmse"].mean(), "MAE_ort_C": -cvr["test_mae"].mean(),
                 "n_hucre": len(r), "n_blok": blocks.nunique()})
reg_tab = pd.DataFrame(rows).round(4)
reg_tab.to_csv(f"{T}/lst_regresyon_metrikleri.csv", index=False, encoding="utf-8-sig")
reg_tab

In [ ]:
import shap
from sklearn.model_selection import cross_val_predict
rf_reg = regs["RandomForestRegressor"]
pred = cross_val_predict(rf_reg, Xr, yr, groups=blocks, cv=gkf, n_jobs=1)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].hexbin(yr, pred, gridsize=60, cmap="viridis", mincnt=1, bins="log")
lims = [min(yr.min(), pred.min()), max(yr.max(), pred.max())]
axes[0].plot(lims, lims, "r--", lw=1)
axes[0].set_xlabel("Gerçek LST (°C)"); axes[0].set_ylabel("Tahmin (°C, blok CV)")
axes[0].set_title(f"RF Regressor — R² = {reg_tab.loc[0, 'R2_ort']:.3f}, RMSE = {reg_tab.loc[0, 'RMSE_ort_C']:.2f} °C")

rf_reg.fit(Xr, yr)
joblib.dump(rf_reg, f"{M}/lst_rf_regressor.joblib", compress=3)
Xs = Xr.sample(min(2000, len(Xr)), random_state=config.RANDOM_STATE)
sv = shap.TreeExplainer(rf_reg).shap_values(Xs)
imp = pd.Series(np.abs(sv).mean(axis=0), index=[REG_NAMES[x] for x in REG_X]).sort_values()
direction = pd.Series([np.corrcoef(Xs[x], sv[:, i])[0, 1] for i, x in enumerate(REG_X)],
                      index=[REG_NAMES[x] for x in REG_X])
imp.plot(kind="barh", ax=axes[1], color=["#d73027" if direction[i] > 0 else "#4575b4" for i in imp.index])
axes[1].set_title("SHAP önemi (kırmızı: değer arttıkça LST artıyor, mavi: azalıyor)", fontsize=10)
axes[1].set_xlabel("ortalama |SHAP| (°C)")
savefig("lst_regresyon_shap")
shap_tab = pd.DataFrame({"ortalama_abs_shap_C": imp, "yon_korelasyonu": direction}).sort_values("ortalama_abs_shap_C", ascending=False).round(4)
shap_tab.to_csv(f"{T}/lst_regresyon_shap.csv", encoding="utf-8-sig")
shap_tab

## B6. Kentsel ısı adası (AS2: İstanbul, İzmir)
**Kentsel ısı adası (UHI):** Şehirlerin çevrelerindeki kırsal alanlardan daha sıcak olması. Bunun başlıca nedenleri beton ve asfaltın ısıyı depolaması, bitki örtüsünün az olması ve buharlaşmanın düşük olmasıdır.

**Tanım (1 km hücre):**
- **Kentsel:** yapılaşma oranı > %50
- **Kırsal:** yapılaşma oranı < %5, su oranı < %10 ve rakımı kentsel hücrelerin medyan rakımından en fazla 200 m farklı. Rakım koşulu, dağ serinliğinin ısı adasını olduğundan büyük göstermesini önler.

**UHI şiddeti** = kentsel hücrelerin ortalama LST'si − kırsal hücrelerin ortalama LST'si. Hem Landsat hem MODIS ile, iki yıl için ayrı ayrı hesaplanıyor.

In [ ]:
rows, box = [], []
for city in config.SUB_QUESTIONS["AS2"]:
    for y in config.CHANGE_YEARS:
        d = cells[(cells["city"] == city) & (cells[f"st_valid_{y}"] >= 0.5) & (cells[f"lc_valid_{y}"] > 0.5)]
        urban = d[d[f"lc2_{y}"] > config.UHI_URBAN_MIN]
        elev0 = urban["elevation"].median()
        rural = d[(d[f"lc2_{y}"] < config.UHI_RURAL_MAX) & (d[f"mndwi_water_{y}"] < config.UHI_WATER_MAX)
                  & ((d["elevation"] - elev0).abs() <= config.UHI_ELEV_TOL)]
        for src, col in [("Landsat", f"LST_{y}"), ("MODIS", f"modis_lst_day_{y}")]:
            rows.append({"city": city, "year": y, "kaynak": src, "n_kentsel": len(urban), "n_kirsal": len(rural),
                         "kentsel_LST": urban[col].mean(), "kirsal_LST": rural[col].mean(),
                         "UHI_C": urban[col].mean() - rural[col].mean()})
        box += [{"city": city, "year": y, "tip": "Kentsel", "LST": v} for v in urban[f"LST_{y}"]]
        box += [{"city": city, "year": y, "tip": "Kırsal", "LST": v} for v in rural[f"LST_{y}"]]
uhi = pd.DataFrame(rows).round(3)
uhi.to_csv(f"{T}/kentsel_isi_adasi.csv", index=False, encoding="utf-8-sig")

bx = pd.DataFrame(box)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, city in zip(axes, config.SUB_QUESTIONS["AS2"]):
    sns.boxplot(data=bx[bx["city"] == city], x="year", y="LST", hue="tip", ax=ax, fliersize=1,
                palette={"Kentsel": "#d73027", "Kırsal": "#1b7837"})
    u = uhi[(uhi["city"] == city) & (uhi["kaynak"] == "Landsat")].set_index("year")["UHI_C"]
    ax.set_title(f"{city} — UHI (Landsat): " + ", ".join(f"{y}: {v:+.1f} °C" for y, v in u.items()))
    ax.set_xlabel(""); ax.set_ylabel("Yaz LST (°C)")
savefig("kentsel_isi_adasi")
uhi

## B7. Su yüzeyi değişimi (AS3: Konya, Van, Samsun)
Açık su alanı (MNDWI > 0) 2014 → 2024 değişimi, modelin Su/Sulak sınıfı alanı ve aynı illerin yağış ve toprak nemi trendleri yan yana veriliyor. Faz 3'teki sulak alan bulgusu nedeniyle (otsu sulak alan çoğu zaman çayır/tarım olarak tahmin edilebiliyor) açık su için MNDWI'yi esas alın.

In [ ]:
water = pd.read_csv(f"{T}/su_alani_karsilastirma.csv").set_index("city")
cities3 = config.SUB_QUESTIONS["AS3"]
w = pd.DataFrame({
    f"mndwi_su_{Y0}_km2": water.loc[cities3, f"mndwi_{Y0}_km2"],
    f"mndwi_su_{Y1}_km2": water.loc[cities3, f"mndwi_{Y1}_km2"],
    "mndwi_degisim_yuzde": water.loc[cities3, "mndwi_degisim_yuzde_2014_2024"],
    "jrc_mndwi_iou_2021": water.loc[cities3, "iou_2021"],
    "model_su_sulak_degisim_km2": areas[(areas["sinif"] == config.CLASSES[5])].set_index("city").loc[cities3, "degisim_km2"],
})
for v in ["chirps_precip_annual", "era5_sm_summer"]:
    t_ = trend[trend["degisken"] == v].set_index("city")
    w[f"{v}_sen_10yil"] = t_.loc[cities3, "sen_egimi_10yil"]
    w[f"{v}_p"] = t_.loc[cities3, "p"]
w = w.round(4)
w.to_csv(f"{T}/su_degisim_ozet.csv", encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 4.5))
w[[f"mndwi_su_{Y0}_km2", f"mndwi_su_{Y1}_km2"]].plot(kind="bar", ax=ax, color=["#9ecae1", "#08519c"], width=0.7)
for i, v in enumerate(w["mndwi_degisim_yuzde"]):
    ax.text(i, w.iloc[i, :2].max(), f"{v:+.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Açık su alanı (km², MNDWI > 0)"); ax.set_title(f"Su yüzeyi {Y0} → {Y1}"); ax.tick_params(axis="x", rotation=0)
savefig("su_alani_degisim")
w

## B8. Alt soru özet tablosu
Her alt soru için ilgili sayısal sonuçlar ve grafik dosyaları otomatik dolduruluyor. **Bulgu** sütunu yalnızca ölçülen sayıyı tarif eder; yorumu siz yazacaksınız.

In [ ]:
def sp_val(city, drv, tgt):
    q = sp[(sp["city"] == city) & (sp["degisken"] == drv) & (sp["hedef"] == tgt)]
    return (f"ρ = {q['spearman_r'].iloc[0]:+.2f} (p = {q['p'].iloc[0]:.3g}, n = {q['n_hucre'].iloc[0]})"
            if len(q) else "hesaplanamadı")

def pp_val(city, cls):
    q = areas[(areas["city"] == city) & (areas["sinif"] == config.CLASSES[cls])]
    return f"{q['degisim_pp'].iloc[0]:+.2f} pp ({q['degisim_km2'].iloc[0]:+.1f} km²)"

def tr_val(city, v):
    q = trend[(trend["city"] == city) & (trend["degisken"] == v)].iloc[0]
    return f"{q['sen_egimi_10yil']:+.3f}/10 yıl (MK p = {q['p']:.3g})"

L, MO, N = "ΔLST (Landsat)", "ΔLST (MODIS)", "ΔNDMI (nem)"
rows = []
for city in config.SUB_QUESTIONS["AS1"]:
    rows += [("AS1", city, f"Orman payı değişimi {Y0}→{Y1}", pp_val(city, 0), "sinif_degisim_pp.png"),
             ("AS1", city, "Δorman – ΔLST", sp_val(city, "Δorman", L), "spearman_isi_haritasi.png"),
             ("AS1", city, "ΔNDVI – ΔLST", sp_val(city, "ΔNDVI", L), "degisim_iklim_dagilim.png"),
             ("AS1", city, "MODIS yaz gündüz LST trendi", tr_val(city, "modis_lst_day"), "iklim_serileri.png")]
for city in config.SUB_QUESTIONS["AS2"]:
    u = uhi[(uhi["city"] == city) & (uhi["kaynak"] == "Landsat")].set_index("year")["UHI_C"]
    rows += [("AS2", city, f"Yapılaşma payı değişimi {Y0}→{Y1}", pp_val(city, 2), "sinif_degisim_pp.png"),
             ("AS2", city, "Δyapılaşma – ΔLST", sp_val(city, "Δyapılaşma", L), "spearman_isi_haritasi.png"),
             ("AS2", city, "UHI şiddeti (Landsat)", f"{Y0}: {u.get(Y0, np.nan):+.2f} °C, {Y1}: {u.get(Y1, np.nan):+.2f} °C",
              "kentsel_isi_adasi.png")]
for city in config.SUB_QUESTIONS["AS3"]:
    rows += [("AS3", city, f"Açık su alanı değişimi {Y0}→{Y1} (MNDWI)", f"{w.loc[city, 'mndwi_degisim_yuzde']:+.1f}%",
              "su_alani_degisim.png"),
             ("AS3", city, "Δsu – ΔLST", sp_val(city, "Δsu", L), "spearman_isi_haritasi.png"),
             ("AS3", city, "Δsu – ΔNDMI", sp_val(city, "Δsu", N), "spearman_isi_haritasi.png"),
             ("AS3", city, "ERA5 yaz toprak nemi trendi", tr_val(city, "era5_sm_summer"), "iklim_serileri.png"),
             ("AS3", city, "CHIRPS yıllık yağış trendi", tr_val(city, "chirps_precip_annual"), "iklim_serileri.png")]
for city in config.SUB_QUESTIONS["AS4"]:
    rows += [("AS4", city, f"Tarım payı değişimi {Y0}→{Y1}", pp_val(city, 1), "sinif_degisim_pp.png"),
             ("AS4", city, "Δtarım – ΔNDMI", sp_val(city, "Δtarım", N), "spearman_isi_haritasi.png"),
             ("AS4", city, "Δtarım – ΔLST", sp_val(city, "Δtarım", L), "spearman_isi_haritasi.png"),
             ("AS4", city, "ERA5 yaz toprak nemi trendi", tr_val(city, "era5_sm_summer"), "iklim_serileri.png")]
top = shap_tab.index[0]
rows.append(("AS1–AS4", "Tümü", "LST regresyonu (RF, mekânsal blok CV)",
             f"R² = {reg_tab.loc[0, 'R2_ort']:.3f}, RMSE = {reg_tab.loc[0, 'RMSE_ort_C']:.2f} °C, "
             f"MAE = {reg_tab.loc[0, 'MAE_ort_C']:.2f} °C; en etkili: {top}", "lst_regresyon_shap.png"))
summary = pd.DataFrame(rows, columns=["Alt soru", "Şehir", "Bulgu", "Sayısal değer", "Grafik dosyası"])
summary.to_csv(f"{T}/alt_soru_ozet_AS1_AS4.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", None)
summary

## Faz 4 bitti mi?
- [ ] `gee_vs_colab_karsilastirma.csv`: Earth Engine modeli ile Colab modeli aynı test noktalarında
- [ ] `sinif_alanlari_degisim.csv`, `gecis_matrisi.csv`, `degisim_haritalari.png`
- [ ] `iklim_trend_mann_kendall.csv`, `spearman_degisim_iklim.csv`, `lst_regresyon_metrikleri.csv`, `kentsel_isi_adasi.csv`
- [ ] `alt_soru_ozet_AS1_AS4.csv`: her alt soru için en az bir sayı ve grafik

## Bu fazda ne yaptık, neden yaptık
- 2021 etiketiyle eğitilen modeli 2014 ve 2024'e uyguladık. Sınıf alanlarını, değişimi ve geçiş matrisini çıkardık. Model haritalarını Earth Engine'de ürettik (indirme yok), Colab modeliyle aynı test noktalarında karşılaştırıp tutarlılığını gösterdik.
- İklim göstergelerinin 22 yıllık eğilimini Mann-Kendall ve Sen eğimi ile ölçtük. Bu yöntemler normal dağılım varsaymaz ve aykırı yıllara dayanıklıdır.
- Arazi örtüsü değişimi ile sıcaklık ve nem değişimi arasındaki ilişkiyi 1 km hücrelerde Spearman korelasyonuyla ölçtük. Sonuçları neden-sonuç olarak değil, ilişki olarak raporluyoruz.
- LST'yi arazi örtüsü ve indekslerle tahmin eden bir regresyon modeli kurduk. Mekânsal blok CV ile komşu hücre sızıntısını önledik, SHAP ile hangi değişkenin etkili olduğunu gösterdik.
- İstanbul ve İzmir'de kentsel ısı adası şiddetini rakım kontrollü bir kırsal referansla iki yıl için hesapladık.